# RSnowflake -- Feature Demo

An interactive walkthrough of **RSnowflake** (multi-backend DBI connector
for Snowflake) running inside a Snowflake Workspace Notebook.

**Architecture:** REST API v2 (baseline) + optional ADBC acceleration
(Arrow-native reads, PUT+COPY INTO writes) when `adbcsnowflake` is installed.

**Sections:**
1. Setup (R environment + PAT auth)
2. Connect
3. Simple queries & type mapping
4. Grid viewer for R data.frames
5. Table operations (write, read, append)
6. Identifier case handling
7. Parameterized queries
8. Transactions
9. dbplyr / dplyr integration
10. Arrow fast path
11. ADBC backend acceleration (reads and writes)
12. Database object browsing (dbListObjects)
13. Cleanup

## 1. Setup

Install the R environment (skip if already done in this session),
register the `%%R` magic, create a PAT, and push session context to env vars.

In [ ]:
# Install R + rpy2 via setup script (included in this directory)
!bash setup_r_environment.sh --basic

In [ ]:
from r_helpers import setup_r_environment
setup_r_environment()

In [ ]:
%%R
# Install RSnowflake dependencies (no-op if already present)
pkgs <- c("DBI", "httr2", "jsonlite", "rlang", "cli",
          "dbplyr", "dplyr", "nanoarrow")
for (pkg in pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
  }
}
cat("Core dependencies OK.\n")

# Install ADBC packages (optional but recommended for performance)
adbc_repos <- c("https://community.r-multiverse.org", "https://cloud.r-project.org")
if (!requireNamespace("adbcdrivermanager", quietly = TRUE)) {
  cat("Installing adbcdrivermanager...\n")
  install.packages("adbcdrivermanager", repos = adbc_repos)
}
if (!requireNamespace("adbcsnowflake", quietly = TRUE)) {
  cat("Installing adbcsnowflake from r-multiverse...\n")
  install.packages("adbcsnowflake", repos = adbc_repos)
}
cat("adbcdrivermanager:", requireNamespace("adbcdrivermanager", quietly = TRUE), "\n")
cat("adbcsnowflake:    ", requireNamespace("adbcsnowflake", quietly = TRUE), "\n")

In [ ]:
%%R
# Install (or reinstall) RSnowflake from the repo source.
if ("RSnowflake" %in% loadedNamespaces()) {
  try(detach("package:RSnowflake", unload = TRUE), silent = TRUE)
  unloadNamespace("RSnowflake")
  cat("Unloaded previous RSnowflake namespace.\n")
}

pkg_dir <- normalizePath(file.path(getwd(), "..", ".."))
cat("Installing RSnowflake from:", pkg_dir, "\n")
install.packages(pkg_dir, repos = NULL, type = "source")

In [ ]:
import os
from snowflake.snowpark.context import get_active_session
from r_helpers import PATManager

session = get_active_session()
pat_mgr = PATManager(session)

if os.environ.get("SNOWFLAKE_PAT"):
    print("PAT already set in environment -- skipping creation.")
    print(f"SNOWFLAKE_PAT set: True (len={len(os.environ['SNOWFLAKE_PAT'])})")
else:
    result = pat_mgr.create_pat(days_to_expiry=1, force_recreate=True)
    if result['success']:
        print(f"PAT created for {result['user']} (role: {result['role_restriction']})")
        print(f"Expires: {result['expires_at']}")
    else:
        print(f"PAT creation failed: {result['error']}")

In [ ]:
import os

os.environ["SNOWFLAKE_ACCOUNT"]   = session.get_current_account().replace('"', '')
os.environ["SNOWFLAKE_USER"]      = session.sql("SELECT CURRENT_USER()").collect()[0][0]
os.environ["SNOWFLAKE_DATABASE"]  = (session.get_current_database() or "").replace('"', '')
os.environ["SNOWFLAKE_SCHEMA"]    = (session.get_current_schema() or "").replace('"', '')
os.environ["SNOWFLAKE_WAREHOUSE"] = (session.get_current_warehouse() or "").replace('"', '')
os.environ["SNOWFLAKE_ROLE"]      = (session.get_current_role() or "").replace('"', '')

for k in ["SNOWFLAKE_ACCOUNT", "SNOWFLAKE_DATABASE", "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_ROLE"]:
    print(f"{k}: {os.environ[k]}")

## 2. Connect

In [ ]:
%%R
if (!nzchar(Sys.getenv("TZ", ""))) Sys.setenv(TZ = "UTC")
options(width = 200)

library(DBI)
library(RSnowflake)

con <- dbConnect(Snowflake())
con
dbGetInfo(con)

## 3. Simple Queries & Type Mapping

In [ ]:
%%R
dbGetQuery(con, "SELECT CURRENT_VERSION() AS version")

In [ ]:
%%R
dbGetQuery(con, "
  SELECT
    42            AS int_val,
    3.14::DOUBLE  AS dbl_val,
    'hello'       AS str_val,
    TRUE          AS bool_val,
    CURRENT_DATE()          AS date_val,
    CURRENT_TIMESTAMP()     AS ts_val
")

## 4. Grid Viewer for R Data.frames

When a `%%R` cell's last visible expression is a `data.frame` (or tibble),
it is **automatically** converted to a pandas DataFrame and returned to
Workspace, which renders the interactive grid viewer -- just like Python cells.

The query in the previous cell already demonstrates this: the grid appears
automatically because `dbGetQuery()` returns a data.frame.

**Flags:**
- `%%R --text` -- opt out; force plain text output
- `%%R --df varname` -- explicitly name which R variable to display in the grid

In [ ]:
%%R --df result
result <- dbGetQuery(con, "
  SELECT 42 AS INT_VAL, 3.14::DOUBLE AS DBL_VAL, 'hello' AS STR_VAL
")
cat("Query returned", nrow(result), "row(s)\n")
cat("Grid viewer shows the 'result' variable below (via --df flag)")


In [ ]:
%%R --text
dbGetQuery(con, "SELECT 1 AS ID, 'text mode' AS NOTE")

## 5. Table Operations

Write a demo data.frame, read it back, and append more rows.
Column names are uppercased by default (standard Snowflake behaviour).

In [ ]:
%%R
demo <- data.frame(
  id     = 1:10,
  city   = c("London", "Paris", "Tokyo", "Sydney", "NYC",
             "Berlin", "Toronto", "Mumbai", "Seoul", "Dubai"),
  temp_c = c(12.5, 15.2, 22.3, 25.1, 18.7,
             10.3, 8.9, 33.2, 19.8, 38.5),
  rainy  = c(TRUE, TRUE, FALSE, FALSE, TRUE,
             TRUE, TRUE, FALSE, FALSE, FALSE),
  stringsAsFactors = FALSE
)

dbWriteTable(con, "DEMO_CITIES", demo, overwrite = TRUE)
cat("Table created.\n")

# Column names are uppercased by default
dbListFields(con, "DEMO_CITIES")

In [ ]:
%%R
dbReadTable(con, "DEMO_CITIES")

In [ ]:
%%R
extra <- data.frame(
  id = 11:12,
  city = c("Rome", "Cairo"),
  temp_c = c(20.1, 35.0),
  rainy = c(FALSE, FALSE)
)
dbAppendTable(con, "DEMO_CITIES", extra)

dbGetQuery(con, "SELECT COUNT(*) AS n FROM DEMO_CITIES")

## 6. Identifier Case Handling

By default, RSnowflake uppercases table and column names to match
Snowflake convention. In raw SQL you can reference them unquoted
(Snowflake auto-uppercases) or with uppercase quoted identifiers.

In [ ]:
%%R
# Columns are uppercase -- unquoted names work, or use uppercase quoted identifiers
dbGetQuery(con, 'SELECT CITY, TEMP_C FROM DEMO_CITIES WHERE TEMP_C > 25')

In [ ]:
%%R
# dbQuoteIdentifier wraps names in double-quotes
dbQuoteIdentifier(con, "myColumn")

# dbUnquoteIdentifier parses back
dbUnquoteIdentifier(con, SQL('"mydb"."myschema"."mytable"'))

## 7. Parameterized Queries

Use `?` placeholders with `params` or `dbBind`.

In [ ]:
%%R
dbGetQuery(
  con,
  'SELECT * FROM DEMO_CITIES WHERE TEMP_C > ?',
  params = list(30)
)

In [ ]:
%%R
res <- dbSendQuery(con, 'SELECT * FROM DEMO_CITIES WHERE CITY = ?')
dbBind(res, list("Tokyo"))
dbFetch(res)
dbClearResult(res)

## 8. Transactions (not yet supported)

The Snowflake SQL API v2 is stateless per-request, so session-based
transactions (`dbBegin`/`dbCommit`/`dbRollback`) are not yet supported.
This section demonstrates that RSnowflake reports a clear error.

In [ ]:
%%R
# Manual transaction -- expected to fail (SQL API v2 is stateless)
tryCatch(
  dbBegin(con),
  error = function(e) cat("Expected:", conditionMessage(e), "\n")
)

In [ ]:
%%R
# dbWithTransaction -- also expected to fail
tryCatch(
  dbWithTransaction(con, {
    dbExecute(con, "SELECT 1")
  }),
  error = function(e) cat("Expected:", conditionMessage(e), "\n")
)

## 9. dbplyr / dplyr Integration

If `dbplyr` and `dplyr` are available, queries can be composed with
familiar tidyverse verbs and translated to Snowflake SQL lazily.

In [ ]:
%%R
if (requireNamespace("dbplyr", quietly = TRUE) &&
    requireNamespace("dplyr", quietly = TRUE)) {

  library(dplyr)

  cities_tbl <- tbl(con, "DEMO_CITIES")

  # Lazy query -- translated to Snowflake SQL, not executed yet
  hot_cities <- cities_tbl |>
    filter(TEMP_C > 20) |>
    select(CITY, TEMP_C) |>
    arrange(desc(TEMP_C))

  cat("== Generated SQL ==\n")
  show_query(hot_cities)

  cat("\n== Results ==\n")
  print(hot_cities |> collect())

  cat("\n== Aggregation ==\n")
  print(
    cities_tbl |>
      summarise(
        avg_temp = mean(TEMP_C, na.rm = TRUE),
        n_rainy  = sum(as.integer(RAINY), na.rm = TRUE),
        n_cities = n()
      ) |>
      collect()
  )

} else {
  cat("Skipped: install dbplyr and dplyr for this section.\n")
}

## 10. Arrow Fast Path (optional)

If `nanoarrow` is installed, RSnowflake can stream results in Arrow format
for lower overhead on large result sets.

In [ ]:
%%R
if (requireNamespace("nanoarrow", quietly = TRUE)) {
  stream <- dbGetQueryArrow(con, "SELECT * FROM DEMO_CITIES")
  arrow_df <- as.data.frame(stream)
  cat("Arrow result:", nrow(arrow_df), "rows,", ncol(arrow_df), "columns\n")
  arrow_df
} else {
  cat("Skipped: install nanoarrow for Arrow fast path.\n")
}

## 11. ADBC Backend Acceleration

When `adbcsnowflake` is installed, RSnowflake transparently routes queries
through the Snowflake ADBC Go driver for significantly better performance:

- **Reads:** Arrow-native transport (no JSON serialisation overhead)
- **Writes:** PUT + COPY INTO (bulk ingest, not row-by-row INSERT)

The ADBC backend connects to the **public endpoint** using PAT auth
(external-client pattern). In `"auto"` mode (the default), RSnowflake
uses ADBC when available and falls back to REST API v2 otherwise.

| Option | Values | Description |
| --- | --- | --- |
| `RSnowflake.backend` | `"auto"`, `"adbc"`, `"rest"` | Which backend to use for reads |
| `RSnowflake.upload_method` | `"auto"`, `"adbc"`, `"literal"` | Which backend to use for writes |

In [ ]:
%%R
has_adbc <- requireNamespace("adbcsnowflake", quietly = TRUE) &&
            requireNamespace("adbcdrivermanager", quietly = TRUE)
cat("ADBC packages installed:", has_adbc, "\n")

if (has_adbc) {
  # --- ADBC read (Arrow-native) ---
  options(RSnowflake.backend = "adbc")
  t0 <- proc.time()
  adbc_df <- dbGetQuery(con, "
    SELECT SEQ4() AS id, RANDSTR(20, RANDOM()) AS txt,
           UNIFORM(0::FLOAT, 100::FLOAT, RANDOM()) AS score
    FROM TABLE(GENERATOR(ROWCOUNT => 50000))
  ")
  adbc_time <- (proc.time() - t0)[["elapsed"]]

  # --- REST read (JSON) ---
  options(RSnowflake.backend = "rest")
  t0 <- proc.time()
  rest_df <- dbGetQuery(con, "
    SELECT SEQ4() AS id, RANDSTR(20, RANDOM()) AS txt,
           UNIFORM(0::FLOAT, 100::FLOAT, RANDOM()) AS score
    FROM TABLE(GENERATOR(ROWCOUNT => 50000))
  ")
  rest_time <- (proc.time() - t0)[["elapsed"]]

  options(RSnowflake.backend = "auto")

  cat(sprintf("ADBC read:  50K rows in %.2fs\n", adbc_time))
  cat(sprintf("REST read:  50K rows in %.2fs\n", rest_time))
  cat(sprintf("Speedup:    %.1fx\n", rest_time / adbc_time))
} else {
  cat("Skipped: install adbcsnowflake for ADBC acceleration.\n")
}

In [ ]:
%%R
if (has_adbc) {
  # Create test data
  write_df <- data.frame(
    id = 1:5000,
    txt = paste0("row_", 1:5000),
    val = runif(5000),
    stringsAsFactors = FALSE
  )

  # --- ADBC write (PUT + COPY INTO) ---
  options(RSnowflake.upload_method = "adbc")
  t0 <- proc.time()
  dbWriteTable(con, "ADBC_DEMO", write_df, overwrite = TRUE)
  adbc_write <- (proc.time() - t0)[["elapsed"]]

  # --- Literal INSERT (REST API) ---
  options(RSnowflake.upload_method = "literal")
  t0 <- proc.time()
  dbWriteTable(con, "ADBC_DEMO", write_df, overwrite = TRUE)
  lit_write <- (proc.time() - t0)[["elapsed"]]

  options(RSnowflake.upload_method = "auto")

  # Verify round-trip
  back <- dbGetQuery(con, "SELECT COUNT(*) AS n FROM ADBC_DEMO")
  cat(sprintf("ADBC write:    5K rows in %.2fs\n", adbc_write))
  cat(sprintf("Literal write: 5K rows in %.2fs\n", lit_write))
  cat(sprintf("Round-trip:    %d rows in table\n", back$N))
  cat(sprintf("\nNote: ADBC has ~5s fixed overhead but scales much better.\n"))
  cat(sprintf("For 20K+ rows, ADBC is typically 3-4x faster than literal INSERT.\n"))
}

## 12. Database Object Browsing

`dbListObjects` lets you navigate the Snowflake object hierarchy
(databases, schemas, tables) from R. In RStudio/Positron it also
powers the Connections Pane.

In [ ]:
%%R
# Top level: databases
cat("== Databases (first 5) ==\n")
head(dbListObjects(con), 5)

In [ ]:
%%R
# Drill into the current database -> schemas
db <- Sys.getenv("SNOWFLAKE_DATABASE", "")
if (nzchar(db)) {
  cat("== Schemas in", db, "==\n")
  print(dbListObjects(con, prefix = Id(catalog = db)))
}

In [ ]:
%%R
# Drill into PUBLIC schema -> tables
if (nzchar(db)) {
  cat("== Tables in", db, ".PUBLIC ==\n")
  print(dbListObjects(con, prefix = Id(catalog = db, schema = "PUBLIC")))
}

## 13. Cleanup

In [ ]:
%%R
dbRemoveTable(con, "DEMO_CITIES")
tryCatch(dbExecute(con, "DROP TABLE IF EXISTS ADBC_DEMO"), error = function(e) NULL)
dbDisconnect(con)
cat("Done! Tables removed and connection closed.\n")